In [26]:
import pandas as pd

sql = pd.read_csv("SQL_DB.csv", encoding = "utf-8-sig")


In [27]:
sql.columns

Index(['size', 'jibun_address', 'road_address', 'name', 'category', 'latitude',
       'longitude', 'phone', 'business_hours', 'review_count', 'description',
       'facilities', 'parking', 'very_good', 'latest_reviews', 'seat_info',
       'tfidf_features', 'menu'],
      dtype='object')

In [28]:
sql = sql.drop(columns=["jibun_address","description","latest_reviews","tfidf_features"])

In [29]:
initial_count = len(sql)

# Drop rows where 'menu' is null
sql = sql.dropna(subset=['menu'])

# Count rows after dropping
final_count = len(sql)

# Calculate the number of dropped rows
dropped_rows = initial_count - final_count

In [30]:
dropped_rows

7

In [31]:
sql.insert(0, "id", range(1, len(sql) + 1))


In [32]:
sql.head(2)

,id,size,road_address,name,category,latitude,longitude,phone,business_hours,review_count,facilities,parking,very_good,seat_info,menu
0,1,63.02,"서울특별시 마포구 양화로6길 57-12, 1층 (서교동)",라운지목화 합정관,중국식,192669.383195,449648.516083,0507-1469-7338,화: 17:00 - 02:00; 수: 17:00 - 02:00; 목: 17:00 -...,291.0,"['단체 이용 가능', '예약', '무선 인터넷']",주차 불가,"[['""음식이 맛있어요""', 293], ['""인테리어가 멋져요""', 168], ['...","['카운터석', '입식', '바테이블']","[['아보카도크림새우', 19900], ['깐풍기', 18500], ['사천가지튀김..."
1,2,610.52,"서울특별시 마포구 양화로 81, H 스퀘어 102호,103호,104호,105호 (서교동)",탭샵바 합정점,한식,192565.856895,449977.778212,0507-1467-0765,화: 11:00 - 24:00; 수: 11:00 - 24:00; 목: 11:00 -...,131.0,"['와인 페어링', '전문 소믈리에', '포장', '단체 이용 가능', '배달', ...",무료 주차 가능,"[['""술이 다양해요""', 111], ['""음식이 맛있어요""', 99], ['""인테...","['룸', '단체석', '카운터석', '테라스', '좌식', '1인석', '연인석']","[['루꼴라 치즈 떡볶이', 9900]]"


In [33]:
sql.to_csv("SQL_DB.csv",encoding="utf-8-sig")

In [34]:
import json
import ast  # To safely evaluate lists

# ✅ Load CSV
df = pd.read_csv("SQL_DB.csv", encoding= "utf-8-sig")

# ✅ Function to clean `menu` column
def parse_menu(menu_str):
    try:
        if pd.isna(menu_str) or menu_str == "":  
            return "[]"  # Store empty list as string

        if isinstance(menu_str, list):  
            return json.dumps(menu_str)  # Convert Python list to JSON string

        # Convert Python-style list to JSON format
        menu_str = menu_str.replace("'", '"').replace("None", "null")  

        # Try JSON parsing first
        return json.dumps(json.loads(menu_str))
    
    except json.JSONDecodeError:
        try:
            return json.dumps(ast.literal_eval(menu_str))  # Try `ast` if JSON fails
        except (ValueError, SyntaxError):
            return "[]"  # If all fails, store empty list

# ✅ Apply function to `menu`
df["menu"] = df["menu"].apply(parse_menu)



df["menu"] = df["menu"].apply(lambda x: json.dumps(json.loads(x), ensure_ascii=False))
df.to_csv("SQL_DB.csv", encoding="utf-8-sig", index=False)

0    [["\uc544\ubcf4\uce74\ub3c4\ud06c\ub9bc\uc0c8\...
1    [["\ub8e8\uaf34\ub77c \uce58\uc988 \ub5a1\ubcf...
2    [["\uba85\ud0dc\ud68c\ub0c9\uba74", 10000], ["...
3    [["\uc720\uc544\ud558 \ud560\uba38\ub2c8\uc758...
4    [["\uaf4c\ud0d5\ubc14\uc624(\uc721\uc999 \uc0e...
Name: menu, dtype: object


In [35]:
print(df["menu"].head())  # Check cleaned menu column

0    [["아보카도크림새우", 19900], ["깐풍기", 18500], ["사천가지튀김...
1                               [["루꼴라 치즈 떡볶이", 9900]]
2    [["명태회냉면", 10000], ["물냉면", 8000], ["비빔냉면", 800...
3    [["유아하 할머니의 훠궈", 32900], ["우삼겹 후추탕", 25000], [...
4    [["꽌탕바오(육즙 샤오롱바오)", 7300], ["성젠바오(육즙 군만두)", 70...
Name: menu, dtype: object
